In [1]:
import pandas as pd
import os
import cv2
import numpy as np
import glob
import shutil
from tqdm.auto import tqdm


/home/quocthai/miniconda3/envs/color-calibration/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# Color Configurations and Nominal Coordinates
# ============================================================

REF = {
    'dark_blue': (30, 26, 120),
    'yellow':    (245, 225, 30),
    'magenta':   (210, 25, 100),
    'green':     (10, 150, 60),
    'red':       (195, 45, 35),
    'blue':      (35, 105, 215)
}

SOLID = {
    (0, 1): 'dark_blue',
    (0, 2): 'yellow',
    (1, 0): 'magenta',
    (2, 0): 'green',
    (3, 1): 'red',
    (3, 2): 'blue'
}

COLOR_RANGES = {
    'yellow':    {'hue': (15, 45),   'saturation': (50, 255)},
    'magenta':   {'hue': (140, 180), 'saturation': (50, 255)},
    'cyan':      {'hue': (90, 120),  'saturation': (50, 255)},
    'green':     {'hue': (45, 90),   'saturation': (50, 255)},
    'red':       {'hue': (160, 25),  'saturation': (50, 255)},
    'dark_blue': {'hue': (110, 140), 'saturation': (30, 255)}
}

NOMINAL = {
    'yellow':    (2.5, 0.5),
    'magenta':   (0.5, 1.5),
    'cyan':      (2.5, 3.5),
    'green':     (0.5, 2.5),
    'red':       (1.5, 3.5),
    'dark_blue': (1.5, 0.5)
}

ROT_CONST = {
    0: None,
    1: cv2.ROTATE_90_COUNTERCLOCKWISE,
    2: cv2.ROTATE_180,
    3: cv2.ROTATE_90_CLOCKWISE
}



In [3]:
# ============================================================
# Helper Functions (Replacing need for wb_hybrid module)
# ============================================================

def gray_world(img):
    b, g, r = cv2.split(img.astype(np.float32))
    b_mean, g_mean, r_mean = b.mean(), g.mean(), r.mean()
    k = (b_mean + g_mean + r_mean) / 3.0
    b_new = np.clip(b * (k / max(1.0, b_mean)), 0, 255)
    g_new = np.clip(g * (k / max(1.0, g_mean)), 0, 255)
    r_new = np.clip(r * (k / max(1.0, r_mean)), 0, 255)
    return cv2.merge([b_new, g_new, r_new]).astype(np.uint8)

def thr(hsv, color_name, sat_min):
    h, s, v = cv2.split(hsv)
    h_lo, h_hi = COLOR_RANGES[color_name]['hue']
    s_min = min(sat_min, COLOR_RANGES[color_name]['saturation'][0])
    if h_lo > h_hi:
        hue_mask = cv2.bitwise_or(cv2.inRange(h, h_lo, 180), cv2.inRange(h, 0, h_hi))
    else:
        hue_mask = cv2.inRange(h, h_lo, h_hi)
    return cv2.bitwise_and(hue_mask, cv2.inRange(s, s_min, 255))

def contours(mask, image_area, img_w):
    k_size = max(3, int(img_w * 0.012) | 1)
    opened = cv2.morphologyEx(mask, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (k_size, k_size)))
    edges = cv2.Canny(cv2.copyMakeBorder(opened, 1, 1, 1, 1, cv2.BORDER_CONSTANT, value=0), 50, 150)
    cnts, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    accepted = []
    for c in cnts:
        c_mapped = c - 1
        area = cv2.contourArea(c_mapped)
        if area < 0.002 * image_area or area > 0.15 * image_area:
            continue
        peri = cv2.arcLength(c_mapped, closed=True)
        approx = cv2.approxPolyDP(c_mapped, 0.02 * peri, closed=True)
        if 4 <= len(approx) <= 6:
            x, y, w, h = cv2.boundingRect(approx)
            if 0.5 < w / float(h) < 2.0:
                M = cv2.moments(approx)
                cx = M["m10"] / M["m00"] if M["m00"] != 0 else x + w/2.0
                cy = M["m01"] / M["m00"] if M["m00"] != 0 else y + h/2.0
                accepted.append({'approx': approx, 'area': area, 'cx': cx, 'cy': cy})
    return accepted

def sane(H):
    h31, h32, h33 = H[2, 0], H[2, 1], H[2, 2]
    corners = [(0.0, 0.0), (400.0, 0.0), (0.0, 400.0), (400.0, 400.0)]
    w_vals = [h31 * cx + h32 * cy + h33 for cx, cy in corners]
    if any(w <= 0.1 for w in w_vals):
        return False
    if abs(h31) >= 0.0015 or abs(h32) >= 0.0015:
        return False
    det = H[0, 0] * H[1, 1] - H[0, 1] * H[1, 0]
    if abs(det) < 1e-2:
        return False
    return True

def proj(H, x, y):
    pt = np.array([[[x, y]]], dtype=np.float32)
    proj_pt = cv2.perspectiveTransform(pt, H)[0][0]
    return proj_pt[0], proj_pt[1]

def brightness(img, H, cg):
    brightness_sum = 0
    img_h, img_w = img.shape[:2]
    r_outer = 50.0
    r_inner = 22.0
    for cx_nom, cy_nom in cg:
        outer_nom = np.array([
            [[cx_nom - r_outer, cy_nom - r_outer]],
            [[cx_nom + r_outer, cy_nom - r_outer]],
            [[cx_nom + r_outer, cy_nom + r_outer]],
            [[cx_nom - r_outer, cy_nom + r_outer]]
        ], dtype=np.float32)
        outer_proj = cv2.perspectiveTransform(outer_nom, H).reshape(-1, 2).astype(np.int32)
        
        inner_nom = np.array([
            [[cx_nom - r_inner, cy_nom - r_inner]],
            [[cx_nom + r_inner, cy_nom - r_inner]],
            [[cx_nom + r_inner, cy_nom + r_inner]],
            [[cx_nom - r_inner, cy_nom + r_inner]]
        ], dtype=np.float32)
        inner_proj = cv2.perspectiveTransform(inner_nom, H).reshape(-1, 2).astype(np.int32)
        
        mask_outer = np.zeros((img_h, img_w), np.uint8)
        mask_inner = np.zeros((img_h, img_w), np.uint8)
        cv2.fillPoly(mask_outer, [outer_proj], 255)
        cv2.fillPoly(mask_inner, [inner_proj], 255)
        mask = cv2.subtract(mask_outer, mask_inner)
        
        mean_bgr = cv2.mean(img, mask=mask)[:3]
        brightness_sum += sum(mean_bgr) / 3.0
    return brightness_sum / len(cg)



In [4]:
# ============================================================
# Two-Stage Peeling Fallback
# ============================================================

def find_card_hull(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    blur = cv2.GaussianBlur(gray, (7, 7), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    sat = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)[:, :, 1]
    card_like = cv2.bitwise_or(th, cv2.inRange(sat, 80, 255))
    k = max(5, int(w * 0.02) | 1)
    ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    closed = cv2.morphologyEx(card_like, cv2.MORPH_CLOSE, ker, iterations=2)
    cnts, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return np.ones((h, w), np.uint8) * 255
    hull = cv2.convexHull(max(cnts, key=cv2.contourArea))
    mask = np.zeros((h, w), np.uint8)
    cv2.fillPoly(mask, [hull], 255)
    return mask

def get_eroded_mask(mask, peel_px):
    if peel_px <= 0:
        return mask.copy()
    padded = cv2.copyMakeBorder(mask, peel_px, peel_px, peel_px, peel_px, cv2.BORDER_CONSTANT, value=0)
    k = 2 * peel_px + 1
    ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    eroded = cv2.erode(padded, ker)
    return eroded[peel_px:-peel_px, peel_px:-peel_px]

def peel_white(img):
    h, w = img.shape[:2]
    card = find_card_hull(img)
    card_area = np.sum(card == 255)
    card_side = np.sqrt(card_area)
    
    # Stage 1 Card interior erosion (3%)
    card_peel = max(1, int(card_side * 0.030))
    card_interior = get_eroded_mask(card, card_peel)
    
    # Detect raw white mask using percentile-based adaptive threshold
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    _, S, V = cv2.split(hsv)
    neutral_mask = (card_interior == 255) & (S <= 55)
    V_neutral = V[neutral_mask]
    if len(V_neutral) > 0:
        v_max = np.percentile(V_neutral, 99)
        adaptive_vmin = int(np.clip(v_max * 0.60, 80, 115))
    else:
        adaptive_vmin = 110
        
    white = (card_interior == 255) & (V >= adaptive_vmin) & (S <= 55)
    white &= np.all(img <= 254, axis=2)
    wm = (white.astype(np.uint8)) * 255
    wm = cv2.morphologyEx(wm, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)))
    
    # Stage 2 White mask erosion (1%)
    white_peel = max(1, int(card_side * 0.010))
    wm_eroded = get_eroded_mask(wm, white_peel)
    wm_eroded = cv2.morphologyEx(wm_eroded, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)))
    return wm_eroded


In [5]:
# ============================================================
# Alignment and Rotation Detection
# ============================================================

def warp_full(img_bgr):
    h, w = img_bgr.shape[:2]
    src = np.array([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]], dtype=np.float32)
    dst = np.array([[0, 0], [599, 0], [599, 599], [0, 599]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(src, dst)
    return cv2.warpPerspective(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), M, (600, 600))

def cell_rgb(W, r, c):
    ch = 150
    y1, x1 = int(r * ch), int(c * ch)
    cell = W[y1 + 40:y1 + 110, x1 + 40:x1 + 110].reshape(-1, 3).astype(float)
    med = np.median(cell, 0)
    d = np.linalg.norm(cell - med, axis=1)
    return cell[d <= np.percentile(d, 70)].mean(0)

def de(a, b):
    a_lab = cv2.cvtColor(np.uint8([[a]]), cv2.COLOR_RGB2Lab)[0, 0].astype(float)
    b_lab = cv2.cvtColor(np.uint8([[np.array(b)]]), cv2.COLOR_RGB2Lab)[0, 0].astype(float)
    return np.linalg.norm(a_lab - b_lab)

def rot_pos(r, c, k):
    for _ in range(k):
        r, c = c, 3 - r
    return r, c

def detect_rotation(img_bgr):
    W = warp_full(img_bgr)
    scores = []
    for k in range(4):
        s = 0
        for (r, c), cname in SOLID.items():
            rr, cc = rot_pos(r, c, k)
            s += de(cell_rgb(W, rr, cc), REF[cname])
        scores.append(s)
    return int(np.argmin(scores)), scores


In [6]:
# ============================================================
# Main Processing Logic (User's Script combined and improved)
# ============================================================

def purify(img, mask, s_max=70, v_floor=185):
    """Keep only pure white pixels: low S, high V (high floor to exclude gray), not saturated."""
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    H, S, V = cv2.split(hsv)
    m = mask > 0
    if m.sum() < 20:
        return np.zeros_like(mask)
    Vm = V[m]
    
    std_v = np.std(Vm)
    if std_v < 15.0:
        # Uniform lighting: Otsu is harmful as it splits a unimodal white distribution.
        otsu = v_floor
    else:
        try:
            otsu, _ = cv2.threshold(Vm, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        except:
            otsu = v_floor
            
    vmin = max(v_floor, int(otsu), int(np.percentile(Vm, 90) * 0.85))
    pure = m & (V >= vmin) & (S <= s_max) & np.all(img <= 254, axis=2)
    return (pure.astype(np.uint8)) * 255

def homography_white_core(img, raw, MIN_HITS=5, core_erode_frac=0.30):
    """
    Project the white regions based on Homography, eroding checkerboard cores
    to stay clear of the black boundaries.
    """
    ih, iw = img.shape[:2]
    area = ih * iw
    one = area / 16
    hsv = cv2.cvtColor(cv2.GaussianBlur(gray_world(img), (5, 5), 0), cv2.COLOR_BGR2HSV)
    cand = []
    for n in COLOR_RANGES:
        for sat in [80, 55]:
            cs = contours(thr(hsv, n, sat), area, iw)
            if cs:
                cand += cs
                break
    uniq = []
    for cd in cand:
        d = False
        for up in uniq:
            if np.hypot(cd['cx'] - up['cx'], cd['cy'] - up['cy']) < 15:
                d = True
                if abs(cd['area'] - one) < abs(up['area'] - one):
                    up.update(cd)
                break
        if not d:
            uniq.append(cd)
    uniq = [u for u in uniq if 0.4 * one <= u['area'] <= 1.6 * one]
    if len(uniq) < MIN_HITS:
        return None, None, f"colors={len(uniq)}<{MIN_HITS}"
    uniq.sort(key=lambda u: abs(u['area'] - one))
    uniq = uniq[:8]
    
    import itertools
    names = list(NOMINAL.keys())
    best = None
    for K in range(min(len(uniq), 6), MIN_HITS - 1, -1):
        for ps in itertools.combinations(uniq, K):
            for cs in itertools.permutations(names, K):
                gp = np.array([[NOMINAL[c][0] * 100, NOMINAL[c][1] * 100] for c in cs], np.float32)
                ip = np.array([[p['cx'], p['cy']] for p in ps], np.float32)
                H, _ = cv2.findHomography(gp, ip)
                if H is None or not sane(H):
                    continue
                fit = np.mean([np.hypot(*np.subtract(proj(H, gp[i][0], gp[i][1]), (ps[i]['cx'], ps[i]['cy']))) for i in range(K)])
                if fit > 10.0:
                    continue
                hits = sum(any(np.hypot(u['cx'] - px, u['cy'] - py) < 0.35 * np.sqrt(one) for u in uniq)
                           for px, py in [proj(H, g[0] * 100, g[1] * 100) for g in NOMINAL.values()])
                sc = (hits, -fit)
                if best is None or sc > best[0]:
                    best = (sc, H)
        if best and best[0][0] >= K:
            break
            
    if best is None or best[0][0] < MIN_HITS:
        return None, None, f"hits<{MIN_HITS}"
        
    H = best[1]
    R180 = np.array([[-1, 0, 400], [0, -1, 400], [0, 0, 1]], np.float32)
    Halt = H @ R180
    cg = [(50., 50.), (50., 350.)]
    finalH = H if brightness(img, H, cg) > brightness(img, Halt, cg) else Halt

    # ---- CHECKER: co mỗi ô trắng vào tâm (lõi) ----
    mask_ck = np.zeros((ih, iw), np.uint8)
    sq = 33.33
    e = core_erode_frac * sq  # co mỗi cạnh e px (toạ độ grid)
    # Loop over all 3x5 grid elements and select checkerboard white squares (c+r is odd)
    for c, r in [(c, r) for r in range(5) for c in range(3) if (c + r) % 2 != 0]:
        xn = 300 + c * sq + e
        yn = 116.67 + r * sq + e
        s2 = sq - 2 * e
        pn = np.array([[[xn, yn]], [[xn + s2, yn]], [[xn + s2, yn + s2]], [[xn, yn + s2]]], np.float32)
        pp = cv2.perspectiveTransform(pn, finalH).reshape(-1, 2).astype(np.int32)
        m = np.zeros((ih, iw), np.uint8)
        cv2.fillPoly(m, [pp], 255)
        mask_ck = cv2.bitwise_or(mask_ck, m)

    # ---- CORNER: vành trắng quanh ô xám (giữ như cũ, có erode nhẹ) ----
    mask_co = np.zeros((ih, iw), np.uint8)
    ro, ri = 50., 22.
    for cx, cy in cg:
        on = np.array([[[cx - ro, cy - ro]], [[cx + ro, cy - ro]], [[cx + ro, cy + ro]], [[cx - ro, cy + ro]]], np.float32)
        inn = np.array([[[cx - ri, cy - ri]], [[cx + ri, cy - ri]], [[cx + ri, cy + ri]], [[cx - ri, cy + ri]]], np.float32)
        op = cv2.perspectiveTransform(on, finalH).reshape(-1, 2).astype(np.int32)
        ip = cv2.perspectiveTransform(inn, finalH).reshape(-1, 2).astype(np.int32)
        mo = np.zeros((ih, iw), np.uint8)
        mi = np.zeros((ih, iw), np.uint8)
        cv2.fillPoly(mo, [op], 255)
        cv2.fillPoly(mi, [ip], 255)
        m = cv2.subtract(mo, mi)
        m = cv2.erode(m, np.ones((3, 3), np.uint8))
        mask_co = cv2.bitwise_or(mask_co, m)

    return mask_ck, mask_co, f"homography hits={best[0][0]}"

def block_means(img, white_bool, n=8):
    h, w = img.shape[:2]
    means = []
    for by in range(n):
        for bx in range(n):
            y0, y1 = by * h // n, (by + 1) * h // n
            x0, x1 = bx * w // n, (bx + 1) * w // n
            sub_m = white_bool[y0:y1, x0:x1]
            if sub_m.sum() < 15:
                continue
            means.append(img[y0:y1, x0:x1][sub_m].mean(axis=0))
    return np.array(means) if means else np.empty((0, 3))

def apply_wb(img, ref):
    B, G, R = ref
    if min(B, G, R) <= 0:
        return img.copy()
    o = img.astype(np.float32)
    o[:, :, 0] *= 255 / B
    o[:, :, 1] *= 255 / G
    o[:, :, 2] *= 255 / R
    return np.clip(o, 0, 255).astype(np.uint8)

def align_and_wb_pipeline(img_path, output_dir):
    """
    Main pipeline to align the image, detect white patches (homography with peeling fallback),
    purify them, calculate white balance, and save visualizations.
    """
    raw = cv2.imread(img_path)
    if raw is None:
        print(f"[-] Cannot read: {img_path}")
        return
        
    base = os.path.splitext(os.path.basename(img_path))[0]
    
    # 1. Detect rotation k and align image
    k, scores = detect_rotation(raw)
    rc = ROT_CONST[k]
    aligned_img = raw.copy() if rc is None else cv2.rotate(raw, rc)
    
    # 2. Detect checkerboard and corner masks
    mck, mco, info = homography_white_core(aligned_img, aligned_img, MIN_HITS=5)
    method = "homography"
    
    if mck is None:
        # Fallback peel: split into right half (checkerboard) and left half (corners)
        wmp = peel_white(aligned_img)
        h, w = aligned_img.shape[:2]
        right = np.zeros_like(wmp)
        right[:, int(w * 0.55):] = 255
        left = np.zeros_like(wmp)
        left[:, :int(w * 0.45)] = 255
        mck = cv2.bitwise_and(wmp, right)
        mco = cv2.bitwise_and(wmp, left)
        method = f"peel ({info})"
        
    # 3. Purify checkerboard and corner masks separately
    pck = purify(aligned_img, mck)
    pco = purify(aligned_img, mco)
    white = (pck > 0) | (pco > 0)
    
    # Lower v_floor fallback if too few white pixels are found
    if white.sum() < 80:
        pck = purify(aligned_img, mck, v_floor=165)
        pco = purify(aligned_img, mco, v_floor=165)
        white = (pck > 0) | (pco > 0)
        
    # 4. Apply White Balance (Global, Region, Median)
    if white.sum() >= 20:
        px = aligned_img[white].astype(np.float32)
        ref_global = px.mean(axis=0)
        
        # Region-based mean
        blk = block_means(aligned_img, white)
        ref_region = blk.mean(axis=0) if blk.shape[0] else ref_global
        
        # Median
        ref_median = np.median(px, axis=0)
    else:
        ref_global = np.array([255.0, 255.0, 255.0])
        ref_region = np.array([255.0, 255.0, 255.0])
        ref_median = np.array([255.0, 255.0, 255.0])
        
    wb_global = apply_wb(aligned_img, ref_global)
    wb_region = apply_wb(aligned_img, ref_region)
    wb_median = apply_wb(aligned_img, ref_median)
    
    # 5. Connected components and pixel stats
    hsv = cv2.cvtColor(aligned_img, cv2.COLOR_BGR2HSV)
    V = hsv[:, :, 2]
    ck_dark = int((V[pck > 0] < 150).sum()) if (pck > 0).any() else 0
    ck_px = int((pck > 0).sum())
    co_px = int((pco > 0).sum())
    
    # 6. Visualization (5-panel montage)
    vis = aligned_img.copy()
    ov = aligned_img.copy()
    ov[pck > 0] = (0, 255, 0)       # checkerboard cores in Green
    ov[pco > 0] = (255, 0, 255)     # corner rings in Magenta
    vis = cv2.addWeighted(vis, 0.55, ov, 0.45, 0)
    
    # Draw red borders around detected patches
    cnts_ck, _ = cv2.findContours(pck, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnts_co, _ = cv2.findContours(pco, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(vis, cnts_ck, -1, (0, 0, 255), 2)  # Red border in BGR
    cv2.drawContours(vis, cnts_co, -1, (0, 0, 255), 2)  # Red border in BGR
    
    panels = [vis, aligned_img, wb_global, wb_region, wb_median]
    labels = ["pure (ck=green,corner=magenta)", "original (aligned)", "WB global", "WB region", "WB median"]
    
    mont = np.hstack([cv2.resize(p, (360, 360)) for p in panels])
    for i, l in enumerate(labels):
        cv2.putText(mont, l, (i * 360 + 6, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
        
    # Save outputs
    cv2.imwrite(os.path.join(output_dir, f"{base}.jpg"), wb_median)


In [7]:
dataset_dir = "../datasets/images"
output_dir = "./images_wb_median"
csv_path = "../datasets/chd_jaundice_pure2.csv"

os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(csv_path)
image_paths = []
for filename in df['image_idx']:
    path = os.path.join(dataset_dir, filename)
    if os.path.exists(path):
        image_paths.append(path)
    else:
        print(f"⚠️ File not found: {path}")

if not image_paths:
    print(f"⚠️ Not found any image listed in CSV in {dataset_dir}")
else:
    print(f"✅ Found {len(image_paths)} images listed in CSV at {dataset_dir}\n")
    
    # ==================== DÙNG TQDM ====================
    for f in tqdm(sorted(image_paths), 
                    desc="progrsss", 
                    unit="image",
                    colour="green",           # màu thanh tiến trình
                    ncols=100):               # độ rộng thanh
        align_and_wb_pipeline(f, output_dir)
    # ==================================================

✅ Found 1805 images listed in CSV at ../datasets/images



progrsss: 100%|██████████████████████████████████████████████| 1805/1805 [08:26<00:00,  3.57image/s]
